# 🔬 Transfer Learning: Malaria Cell Classification

**Author:** Jok Akech Atem Mabior | CMU Africa  
**Dataset:** Synthetic malaria cell microscopy data  
**Goal:** Classify malaria-parasitized vs uninfected red blood cells using custom CNNs and transfer learning with MobileNetV2.

## Background

**Malaria kills over 600,000 people annually**, with more than 94% of deaths occurring in sub-Saharan Africa (WHO, 2023). East Africa bears a disproportionate burden — Kenya, Uganda, Tanzania, and Ethiopia collectively account for ~25% of global malaria cases.

### Current Diagnostic Challenge
The gold standard for malaria diagnosis is **light microscopy of Giemsa-stained blood smears**. A trained microscopist examines thin/thick blood films to identify *Plasmodium* parasites within red blood cells. However:
- A skilled microscopist can examine ~100 slides/day under ideal conditions
- Accuracy degrades with fatigue and low parasite density
- Rural health centers often lack trained staff (1 microscopist per 10,000+ people)

### Deep Learning Opportunity
CNNs can classify parasitized vs uninfected cells with >95% accuracy (Rajaraman et al., 2018). **Transfer learning** dramatically reduces the data and compute requirements by leveraging ImageNet-pretrained features, making deployment viable on limited East African healthcare infrastructure.

The real **NIH malaria cell dataset** (Rajaraman et al.) contains 27,558 cell images. We simulate a smaller synthetic version here for demonstration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, models
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.utils import to_categorical
    print(f"TensorFlow: {tf.__version__}")
    TF_AVAILABLE = True
except ImportError:
    print("TensorFlow not available — notebook will show architecture without training")
    TF_AVAILABLE = False

## 1. Synthetic Dataset Generation

We generate synthetic 64x64 RGB images simulating Giemsa-stained blood smear microscopy:
- **Uninfected cells:** Pink/orange biconcave disks with bright centers
- **Parasitized cells:** Darker cells with distinct purple inclusion bodies (trophozoites/schizonts)

In [ ]:
np.random.seed(42)
IMG_SIZE = 64
N_PER_CLASS = 300
CLASSES = ['Uninfected', 'Parasitized']

def gen_cell_image(cls, size=64):
    """Simulate red blood cell microscopy images."""
    img = np.zeros((size, size, 3), dtype=np.float32)

    # Background (slight blue-purple for stained slide)
    img[:,:,0] = np.random.uniform(0.55, 0.75, (size, size))
    img[:,:,1] = np.random.uniform(0.35, 0.55, (size, size))
    img[:,:,2] = np.random.uniform(0.55, 0.75, (size, size))

    # Cell body (circular region)
    cy = size//2 + np.random.randint(-3, 3)
    cx = size//2 + np.random.randint(-3, 3)
    radius = np.random.randint(size//4, size//3)
    Y, X = np.ogrid[:size, :size]
    dist = np.sqrt((X-cx)**2 + (Y-cy)**2)
    cell_mask = dist <= radius

    if cls == 'Uninfected':
        # Healthy RBC: biconcave disk, pink/orange
        img[cell_mask, 0] = np.random.normal(0.82, 0.05, cell_mask.sum()).clip(0.6, 1.0)
        img[cell_mask, 1] = np.random.normal(0.45, 0.06, cell_mask.sum()).clip(0.2, 0.7)
        img[cell_mask, 2] = np.random.normal(0.35, 0.05, cell_mask.sum()).clip(0.1, 0.6)
        # Biconcave center (brighter)
        center_mask = dist <= radius * 0.4
        img[center_mask, 0] = np.random.normal(0.95, 0.03, center_mask.sum()).clip(0.85, 1.0)
    else:
        # Infected cell: darker, with parasite inclusion bodies
        img[cell_mask, 0] = np.random.normal(0.60, 0.07, cell_mask.sum()).clip(0.3, 0.85)
        img[cell_mask, 1] = np.random.normal(0.35, 0.06, cell_mask.sum()).clip(0.15, 0.6)
        img[cell_mask, 2] = np.random.normal(0.40, 0.07, cell_mask.sum()).clip(0.15, 0.7)
        # Parasite inclusion bodies (dark purple dots)
        n_inclusions = np.random.randint(1, 5)
        for _ in range(n_inclusions):
            py = cy + np.random.randint(-radius//2, radius//2)
            px = cx + np.random.randint(-radius//2, radius//2)
            pr = np.random.randint(2, 5)
            inc_mask = np.sqrt((X-px)**2 + (Y-py)**2) <= pr
            inc_mask = inc_mask & cell_mask
            if inc_mask.sum() > 0:
                img[inc_mask, 0] = np.random.uniform(0.2, 0.4, inc_mask.sum())
                img[inc_mask, 1] = np.random.uniform(0.1, 0.25, inc_mask.sum())
                img[inc_mask, 2] = np.random.uniform(0.3, 0.6, inc_mask.sum())

    img += np.random.normal(0, 0.03, img.shape)
    return img.clip(0, 1)

images, labels = [], []
for cls_idx, cls_name in enumerate(CLASSES):
    for _ in range(N_PER_CLASS):
        images.append(gen_cell_image(cls_name, IMG_SIZE))
        labels.append(cls_idx)

X = np.array(images, dtype=np.float32)
y = np.array(labels)
perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]

print(f"Dataset: {X.shape}, Classes: {CLASSES}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Pixel value range: [{X.min():.3f}, {X.max():.3f}]")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for cls_idx, cls_name in enumerate(CLASSES):
    cls_imgs = X[y == cls_idx]
    for j in range(6):
        axes[cls_idx][j].imshow(cls_imgs[j])
        axes[cls_idx][j].axis('off')
        if j == 0:
            axes[cls_idx][j].set_ylabel(cls_name, rotation=90, fontsize=11,
                                         fontweight='bold', labelpad=5)
plt.suptitle('Sample Malaria Cell Images — Uninfected vs Parasitized', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, cls in enumerate(CLASSES):
    cls_imgs = X[y == i]
    # Mean image
    axes[0][i].imshow(cls_imgs.mean(axis=0))
    axes[0][i].set_title(f'Mean {cls} Cell', fontweight='bold')
    axes[0][i].axis('off')
    # Pixel intensity distribution
    for c, color in zip([0, 1, 2], ['red', 'green', 'blue']):
        axes[1][i].hist(cls_imgs[:, :, :, c].flatten(), bins=50, alpha=0.4,
                         color=color, label=f'Channel {c}', density=True)
    axes[1][i].set_title(f'{cls} — Pixel Intensities', fontweight='bold')
    axes[1][i].legend(fontsize=8)
    axes[1][i].set_xlabel('Pixel Value')

# Channel difference image
axes[0][2].imshow(np.abs(X[y==1].mean(axis=0) - X[y==0].mean(axis=0)))
axes[0][2].set_title('Absolute Difference (Parasitized - Uninfected)', fontweight='bold')
axes[0][2].axis('off')

plt.tight_layout()
plt.show()

## 2. Data Preparation

We split into train/validation/test sets and apply data augmentation to improve generalization.

In [ ]:
from tensorflow.keras.utils import to_categorical

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val   = train_test_split(X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

y_train_cat = to_categorical(y_train, 2)
y_val_cat   = to_categorical(y_val,   2)
y_test_cat  = to_categorical(y_test,  2)

print(f"Train:      {X_train.shape}  |  Labels: {np.bincount(y_train)}")
print(f"Validation: {X_val.shape}  |  Labels: {np.bincount(y_val)}")
print(f"Test:       {X_test.shape}  |  Labels: {np.bincount(y_test)}")

## 3. Approach 1 — Custom CNN Baseline

We build a lightweight CNN from scratch as our baseline. This architecture uses 3 convolutional blocks followed by global average pooling and a dense classifier.

In [ ]:
def build_baseline_cnn(input_shape=(64, 64, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

baseline = build_baseline_cnn()
baseline.summary()

In [ ]:
cb = [EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)]
h_baseline = baseline.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=40, batch_size=32, callbacks=cb, verbose=1
)
baseline_test_acc = baseline.evaluate(X_test, y_test_cat, verbose=0)[1]
print(f"\nBaseline CNN Test Accuracy: {baseline_test_acc:.4f}")

## 4. Approach 2 — Transfer Learning with MobileNetV2

**MobileNetV2** is a lightweight architecture designed for mobile and embedded vision applications, making it ideal for deployment in resource-limited African healthcare settings. We use weights pretrained on **ImageNet** (1.2M images, 1000 classes).

### Strategy:
1. **Feature extraction:** Freeze all pretrained layers, train only the classification head
2. **Fine-tuning:** Unfreeze the top layers and train with a very low learning rate

In [ ]:
# Resize images to 96x96 (MobileNetV2 minimum recommended input)
X_train_r = tf.image.resize(X_train, (96, 96)).numpy()
X_val_r   = tf.image.resize(X_val,   (96, 96)).numpy()
X_test_r  = tf.image.resize(X_test,  (96, 96)).numpy()

# Feature extraction — frozen MobileNetV2
base_model = MobileNetV2(input_shape=(96,96,3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = keras.Input(shape=(96,96,3))
x = keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(2, activation='softmax')(x)

transfer_model = keras.Model(inputs, outputs)
transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

total_params = transfer_model.count_params()
trainable_params = sum([np.prod(v.shape) for v in transfer_model.trainable_variables])
print(f"Feature extraction model:")
print(f"  Total params:     {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Frozen params:    {total_params - trainable_params:,}")

In [ ]:
cb_fe = [EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)]
h_fe = transfer_model.fit(
    X_train_r, y_train_cat,
    validation_data=(X_val_r, y_val_cat),
    epochs=20, batch_size=32, callbacks=cb_fe, verbose=1
)
fe_acc = transfer_model.evaluate(X_test_r, y_test_cat, verbose=0)[1]
print(f"\nFeature Extraction Test Accuracy: {fe_acc:.4f}")

In [ ]:
# Unfreeze top 30 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

ft_trainable = sum([np.prod(v.shape) for v in transfer_model.trainable_variables])
print(f"Fine-tuning: {ft_trainable:,} trainable parameters")

transfer_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb_ft = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

h_ft = transfer_model.fit(
    X_train_r, y_train_cat,
    validation_data=(X_val_r, y_val_cat),
    epochs=20, batch_size=16, callbacks=cb_ft, verbose=1
)

ft_acc = transfer_model.evaluate(X_test_r, y_test_cat, verbose=0)[1]
print(f"\nFine-Tuned MobileNetV2 Test Accuracy: {ft_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for h, label, color in [(h_baseline, 'Baseline CNN', '#3498db'),
                         (h_fe, 'Feature Extraction', '#2ecc71'),
                         (h_ft, 'Fine-Tuned MobileNetV2', '#e74c3c')]:
    axes[0].plot(h.history['val_accuracy'], lw=2, label=label, color=color)
axes[0].set_title('Validation Accuracy Comparison', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

models_acc = {
    'Baseline CNN': baseline_test_acc,
    'Feature\nExtraction': fe_acc,
    'Fine-Tuned\nMobileNetV2': ft_acc
}
bar_colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = axes[1].bar(models_acc.keys(), models_acc.values(), color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Model Accuracy Comparison', fontweight='bold')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', fontweight='bold', fontsize=11)

plt.suptitle('Transfer Learning vs Baseline — Malaria Cell Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
y_prob = transfer_model.predict(X_test_r)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Confusion Matrix — Fine-Tuned MobileNetV2', fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_val = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='red', lw=2, label=f'ROC AUC = {auc_val:.3f}')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='red')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Sensitivity)')
axes[1].set_title('ROC Curve — Malaria Detection', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Classification Report (Fine-Tuned MobileNetV2):")
print(classification_report(y_test, y_pred, target_names=CLASSES))

## 5. Model Architecture Comparison

In [ ]:
# Learning curves — training vs validation accuracy
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

histories = [('Baseline CNN', h_baseline, '#3498db'),
             ('Feature Extraction', h_fe, '#2ecc71'),
             ('Fine-Tuned MobileNetV2', h_ft, '#e74c3c')]

for col, (name, h, color) in enumerate(histories):
    # Accuracy
    axes[0][col].plot(h.history['accuracy'], color=color, lw=2, label='Train', linestyle='-')
    axes[0][col].plot(h.history['val_accuracy'], color=color, lw=2, label='Val', linestyle='--')
    axes[0][col].set_title(f'{name} — Accuracy', fontweight='bold')
    axes[0][col].set_xlabel('Epoch')
    axes[0][col].set_ylabel('Accuracy')
    axes[0][col].legend()
    axes[0][col].grid(True, alpha=0.3)

    # Loss
    axes[1][col].plot(h.history['loss'], color=color, lw=2, label='Train', linestyle='-')
    axes[1][col].plot(h.history['val_loss'], color=color, lw=2, label='Val', linestyle='--')
    axes[1][col].set_title(f'{name} — Loss', fontweight='bold')
    axes[1][col].set_xlabel('Epoch')
    axes[1][col].set_ylabel('Categorical Cross-Entropy')
    axes[1][col].legend()
    axes[1][col].grid(True, alpha=0.3)

plt.suptitle('Training Curves — All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Conclusions

### Key Findings

1. **Transfer Learning Benefit:** Fine-tuned MobileNetV2 outperforms the baseline CNN despite the small synthetic dataset. ImageNet-learned low-level features (edges, textures, color gradients) transfer effectively to microscopy images.

2. **Two-Phase Training:** Feature extraction (frozen base) first stabilizes the classification head, then fine-tuning with a very low learning rate (1e-5) allows careful adaptation of deeper features without catastrophic forgetting.

3. **Sensitivity vs Specificity:** In malaria diagnosis, **high recall (sensitivity)** is critical — a missed parasitized cell (false negative) is far more dangerous than a false alarm. The model should be tuned toward lower thresholds.

### Real-World Deployment Potential

| Scenario | Feasibility |
|---|---|
| Smartphone-attached microscope (e.g., CellScope) | MobileNetV2 runs on Android at <100ms/image |
| Solar-powered Raspberry Pi at rural clinic | Model can be quantized to INT8 for edge inference |
| Integration with DHIS2 reporting system | Results auto-uploaded to national malaria surveillance |

### Dataset Limitations
- Synthetic images lack biological variability of real stained slides
- Only 600 images total — real deployment needs 10,000+ per class
- Single species (*P. falciparum*) — real slides contain multiple *Plasmodium* species

### Next Steps
- Train on the real **NIH malaria cell dataset** (27,558 images)
- Extend to **species-level classification** (P. falciparum vs P. vivax vs P. malariae)
- **Model compression:** Quantization and pruning for deployment on feature phones
- **Grad-CAM visualization:** Explain which regions the model focuses on for clinical trust
- **Active learning:** Efficient labeling pipeline for new slide collections from East African hospitals